# KV-Edit → Modular Diffusers — E2E (training-free editing, precise background preservation)

Masked text editing on FLUX with cached background K/V (KV-Edit, arXiv:2502.17363). Publish PRIVATE `remyxai/kv-edit-flux-modular` → load via `trust_remote_code` → real subject-swap edits → **quantitative validation**: background L1/SSIM on the kept region (should be high) + CLIP edit-direction (the subject should change) + a head-to-head vs FlowEdit (KV-Edit should preserve the background better). Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.


## 1 · Install + GPU + auth


In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf scikit-image


In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)


In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"KVEditBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.KVEditBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"KVEditBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/kv-edit-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + a source image + mask


In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/kv-edit-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "KVEditBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect KVEditBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/fallenshock/FlowEdit/main/inputs/cat.png"  #@param {type:"string"}
try:
    src = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); src=Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((1024,1024)); src.save("src.png")

# default edit mask: a box over the central subject (white = edit, black = keep). Upload your own for real use.
mask = Image.new("L", (1024,1024), 0)
ImageDraw.Draw(mask).rectangle([256, 256, 768, 768], fill=255)
mask.save("mask.png")
print("source | mask:"); display(src.resize((320,320))); display(mask.resize((320,320)))


## 4 · Spike first — no-op when the mask keeps everything

Run `smoke.ipynb` milestone B first if you haven't: an all-keep mask must reconstruct the source (it exercises the exact capture/substitute seam this notebook relies on).


## 5 · The edits (full settings)


In [ ]:
import torch
from PIL import Image
from IPython.display import display
EDITS = [("a cat", "a dog"), ("a cat", "a tiger")]   # (source_prompt, target_prompt)
outs = [("source", src), ("mask", mask.convert("RGB"))]
for sp, tp in EDITS:
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(image="src.png", mask="mask.png", source_prompt=sp, prompt=tp,
              height=1024, width=1024, T_steps=28, guidance_scale=3.5,
              src_guidance_scale=1.0, generator=g).images[0]
    im.save(f"edit_{tp.replace(' ','_')}.png"); outs.append((tp, im)); print("  ✓", tp)
S = 384; W_ = len(outs)*S + (len(outs)+1)*10
row = Image.new("RGB", (W_, S+40), "white")
from PIL import ImageDraw, ImageFont
d = ImageDraw.Draw(row)
try: Ft = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26)
except Exception: Ft = ImageFont.load_default()
for i, (name, im) in enumerate(outs):
    x = 10 + i*(S+10); row.paste(im.resize((S,S)), (x,0)); d.text((x+8, S+6), name, fill="black", font=Ft)
row.save("kvedit_grid.png"); print("source | mask | edits:"); display(row.resize((min(W_,1400), int((S+40)*min(W_,1400)/W_))))


## 6 · Quantitative — background preserved AND target changed


In [ ]:
import numpy as np, torch
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from transformers import CLIPModel, CLIPProcessor

keep = (np.asarray(mask.resize((1024,1024))) < 128)          # True = unedited background
bg_frac = keep.mean()
print(f"background fraction: {bg_frac:.2%}")
src_np = np.asarray(src).astype(np.float32)/255.0

def bg_scores(im):
    a = np.asarray(im.resize((1024,1024))).astype(np.float32)/255.0
    l1 = float(np.abs(a - src_np)[keep].mean())
    ss = float(ssim((a*keep[...,None]).astype(np.float32), (src_np*keep[...,None]).astype(np.float32),
                    channel_axis=2, data_range=1.0))
    return l1, ss

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def sim(img, text):
    b = proc(text=[text], images=[img], return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad(): o = clip(**b)
    ie = o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); te = o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((ie@te.T)[0,0])

print("edit | background L1 (low=kept) | background SSIM (high=kept) | CLIP edit-direction")
ok = True
for sp, tp in EDITS:
    e = Image.open(f"edit_{tp.replace(' ','_')}.png")
    l1, ss = bg_scores(e)
    se, ss0 = sim(e, tp), sim(src, tp)
    print(f"  {tp:>10}:  L1={l1:.4f}  SSIM={ss:.3f}  |  edit={se:.3f} vs source={ss0:.3f} -> {'PASS' if se>ss0 else 'REVIEW'}")
    ok = ok and (se > ss0) and (ss > 0.85)
print("\nbackground preservation (SSIM>0.85 on kept region) + edit-direction:", "PASS" if ok else "REVIEW")


## 7 · Head-to-head vs FlowEdit (background fidelity)

Same source, same edit, same kept region. FlowEdit is structure-preserving but re-synthesizes the background; KV-Edit should show a lower L1 / higher SSIM on the kept region.


In [ ]:
import gc, torch, numpy as np
# free KV-Edit before loading FlowEdit (both hold full FLUX.1-dev)
del pipe; gc.collect(); torch.cuda.empty_cache()

from diffusers import ModularPipeline
fe = ModularPipeline.from_pretrained("remyxai/flowedit-flux-modular", trust_remote_code=True)
assert type(fe.blocks).__name__ == "FlowEditBlock"
fe.load_components(dtype=DT); fe.to(DEV)

print("edit | KV-Edit bg L1/SSIM | FlowEdit bg L1/SSIM  (same kept region; KV-Edit should win)")
for sp, tp in EDITS:
    g = torch.Generator(DEV).manual_seed(0)
    fim = fe(image="src.png", source_prompt=sp, prompt=tp, height=1024, width=1024,
             T_steps=28, src_guidance_scale=1.5, tar_guidance_scale=5.5, n_max=24, n_min=0,
             generator=g).images[0]
    fim.save(f"flowedit_{tp.replace(' ','_')}.png")
    kv_l1, kv_ss = bg_scores(Image.open(f"edit_{tp.replace(' ','_')}.png"))
    fe_l1, fe_ss = bg_scores(fim)
    verdict = "KV-EDIT PRESERVES BETTER" if kv_ss > fe_ss else "REVIEW"
    print(f"  {tp:>10}:  KV-Edit L1={kv_l1:.4f} SSIM={kv_ss:.3f}  |  FlowEdit L1={fe_l1:.4f} SSIM={fe_ss:.3f}  -> {verdict}")


## Verdict
PASS = `loaded block: KVEditBlock` + the subject changes (CLIP edit-direction) + the kept region stays put (SSIM > 0.85) + KV-Edit beats FlowEdit on background SSIM. On pass: human confirms, flip the repo public, add the Colab badge + umbrella collection.
